In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke([HumanMessage(content="지금 몇 시야?")])    # invoke가 가능하다는 것은 llm도 Runnable하다는 뜻.


AIMessage(content='죄송하지만, 현재 시간을 알 수는 없습니다. 하지만 기기의 시계나 인터넷을 통해 확인하실 수 있습니다. 도움이 필요하면 말씀해 주세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 13, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_09148a4f1e', 'id': 'chatcmpl-EY5KGGfVkQMNpT6iIjVIprymO0hXT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a12f25-3a47-7000-a546-59f1f65141a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 33, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_detai

In [4]:
# 현재 시간을 질문: get_current_time() function => function calling
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool   # LLM에 전달할 함수 명세서 생성
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각을 반환하는 함수
    
    Args:
        timezone (str) : 타임존(예: 'Asia/Seoul'). 실제 존재해야 함.
        location (str) : 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨.
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f"{timezone} ({location} 현재 시각 {now})"    # 반환된 값 -> LLM
    return location_and_local_time


In [5]:
# tool이 된 함수는 반드시 invoke로 실행해야 한다.
# 1. LLM에 보내지 않고 테스트 진행 => 반환 => 문자열
# 문자열  'Asia/Seoul (부산 현재 시각 2026-10-12 14:31:02)'
get_current_time.invoke({"timezone":"Asia/Seoul", "location":"부산"})
# @tool하면 get_current_time은 base tool이 되어서 바로 invoke 가능.

'Asia/Seoul (부산 현재 시각 2026-10-12 17:05:32)'

In [6]:
# 2. LLM에 보내고 받아서(tool_call) 테스트 진행. => 반환 => ToolMessage
# get_current_time.invoke(tool_call)

In [7]:
# bind_tools(tools): 모델(LLM)에게 "이런 도구(함수)들을 사용할 수 있다"고 알려주는 명세서 연결
# llm.bind_tools(함수 목록): 함수 목록: [함수, 함수] (@tool 되어있는 함수.)
# bind_tools(tools, tool_choice=None, parallel_tool_calls, strict=None)
# tools : 함수 목록
# tool_choice : 'auto', required, none, 도구 이름
# parallel_tool_call : 병렬. 한 번에 여러 도구 요청 허용 여부(True/False)
# strict : 모델이 json 스키마를 엄격하게 지키게 함(True)

# llm.bind_tools(tools).invoke()
# client -> LLM : tool 명세서 보내야 함.
# LLM -> client : 실행할 함수 목록 받아야 함.

tools=[get_current_time,]    # LLM에 보낼 함수(오브젝트) 목록. @tool된 것을 보내야 함(""감싸지X).
tool_dict = {"get_current_time":get_current_time}   # LLM이 요청한 함수명 받아서 실행할 함수와 매핑해주는 딕셔너리.
llm_with_tools = llm.bind_tools(tools)
# LLM에 보낼 준비만 한 것.

In [8]:
# 사용자의 질문과 도구를 사용해서 언어 모델(LLM) 답변 생성
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?"),
]

response=llm_with_tools.invoke(messages)
messages.append(response)
print(messages)


[SystemMessage(content='너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_17d13f9f7d', 'id': 'chatcmpl-EY5KHrWyHHYP7R4IbrUzQNYH1LnAJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12f25-40a1-7681-9dcc-23f4bfdb8762-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'As

In [9]:
# 함수 실행 후 messages append  llm에는 아직 안 보냄.
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]    # 선택된 함수 지정
    print(tool_call["args"])
    tool_message = selected_tool.invoke(tool_call)  # args 넣어서 함수 실행
    messages.append(tool_message)

messages

{'timezone': 'Asia/Seoul', 'location': 'Busan'}


[SystemMessage(content='너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_17d13f9f7d', 'id': 'chatcmpl-EY5KHrWyHHYP7R4IbrUzQNYH1LnAJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12f25-40a1-7681-9dcc-23f4bfdb8762-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': '

In [10]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2026년 10월 12일 오후 5시 5분 33초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 186, 'total_tokens': 213, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_17d13f9f7d', 'id': 'chatcmpl-EY5KJFnGflN3bfTAkMVtEY5IvEwYE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a12f25-43ba-74c0-a68c-d081b7b0361e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 186, 'output_tokens': 27, 'total_tokens': 213, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [11]:
# 주식 가격 정보
# 유효성, 형식 정의 => 클래스 생성
# def get_yf_stock_history(ticker: str, period: str): ticker: str, period: str 유효성 검증
from pydantic import BaseModel, Field
class StockHistoryInput(BaseModel): # BaseModel 상속
    ticker: str = Field(...,title="주식코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")
    

In [12]:
# 주식 가격 가져오는 함수 선언
import yfinance as yf

@tool
# def get_yf_stock_history(ticker: str, period: str)
def get_yf_stock_history(stock_history_input: StockHistoryInput):
    """주식 종목의 가격 데이터를 조회하는 함수"""   # 함수에 대한 설명. args 줄 필요가 없어짐. pydantic 때문에.
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()
    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {
    "get_current_time":get_current_time,
    "get_yf_stock_history":get_yf_stock_history
}

llm_with_tools=llm.bind_tools(tools)


In [13]:
# LLM이 어떤 함수를 실행해야 하는지 선택한 함수명, args
messages.append(HumanMessage("테슬라는 한 달 전에 비해 주가가 올랐나 내렸나?"))
response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 277, 'total_tokens': 304, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3846bce368', 'id': 'chatcmpl-EY5KLAajJG7SCOlVageJk8i0l7QSE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a12f25-4c93-7ac3-a950-7b2af7108e04-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_Op4xxfJCJo5dOtTFO0C9Mcqs', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 277, 'output_tokens': 27, 'total_tokens

In [14]:
# 함수 실행
# response.tool_calls[0]  # 실행함수 1개라도 원래는 for문 돌리는 게 정석. LLM이 보내는 거라 확정할 수 없음.
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]    # 함수 선택
    tool_message = selected_tool.invoke(tool_call)  # invoke는 args 꺼내서 넣어줌.  함수 실행 결과. => return ToolMessage
    messages.append(tool_message)
    print(tool_message)




content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-09-10 00:00:00-04:00 | 360.6  | 369.21 | 357.68 |  363.56 | 2.96672e+07 |           0 |              0 |\n| 2026-09-11 00:00:00-04:00 | 364.2  | 368.66 | 361.6  |  365.44 | 3.0153e+07  |           0 |              0 |\n| 2026-09-14 00:00:00-04:00 | 359.78 | 367.73 | 357.04 |  358.97 | 3.24772e+07 |           0 |              0 |\n| 2026-09-15 00:00:00-04:00 | 358.75 | 362.39 | 354.05 |  356.58 | 3.03177e+07 |           0 |              0 |\n| 2026-09-16 00:00:00-04:00 | 357.93 | 365.1  | 354.89 |  358.08 | 3.21774e+07 |           0 |              0 |\n| 2026-09-17 00:00:00-04:00 | 367.48 | 374.12 | 363.22 |  366.2  | 3.89241e+07 |           0 |              0 |\n| 2026-09-18 00:00:00-04:00 | 369    | 370.9  | 360.75 |  364.27 | 5.19223e+07 |       

In [15]:
print(llm_with_tools.invoke(messages).content)

한 달 전 테슬라(TSLA)의 주가는 약 $360.6였고, 현재 주가는 약 $382.7입니다. 따라서 테슬라의 주가는 한 달 전에 비해 올랐습니다.


In [22]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇 시야?"),
]

response=llm_with_tools.stream(messages)

is_first=True
for chunk in response:
    print("chunk type: ", type(chunk))

    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk

    print(f"content: {gathered.content}, tool_call_chunk, {gathered.tool_calls}")
messages.append(gathered)


chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {}, 'id': 'call_FM3ZKSmhGAgASs1DfmjJTeWs', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {}, 'id': 'call_FM3ZKSmhGAgASs1DfmjJTeWs', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {}, 'id': 'call_FM3ZKSmhGAgASs1DfmjJTeWs', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_FM3ZKSmhGAgASs1DfmjJTeWs', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content: , tool_call_chunk, [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_FM3ZKSmhGAgASs1DfmjJTeWs', 'type': 't

In [ ]:
# 함수 실행
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
messages

{'timezone': 'Asia/Seoul', 'location': 'Busan'}


[SystemMessage(content='너는 사용자의 질문에 답변하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇 시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3846bce368', 'service_tier': 'default'}, id='lc_run--01a12f2d-447f-7bb3-8f84-770583636833', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': 'Busan'}, 'id': 'call_FM3ZKSmhGAgASs1DfmjJTeWs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 23, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"Busan"}', 'id': 'call_FM3ZKSmhGAgASs1DfmjJTeWs', 'index': 0

In [24]:
for c in llm_with_tools.stream(messages):
    print(c.content, end="|")

|부|산|은| 현재| |202|6|년| |10|월| |12|일| |17|시| |14|분|입니다|.||||